<a href="https://colab.research.google.com/github/MehrdadRS95/AISec/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade torchvision datasets

In [ ]:
!pip install --upgrade datasets

### Note on the ImportError
After running the installation above, you may need to **restart the runtime** (Runtime > Restart session) for the changes to take effect and resolve the `VideoReader` import issue.

In [ ]:
import torch
from torch.utils.data import DataLoader
# Change this:

from transformers import BertTokenizer, BertForSequenceClassification, get_scheduler
from datasets import load_dataset
from torch.optim import AdamW
from tqdm.auto import tqdm

In [ ]:

# ==========================================
# 1. SETUP & HYPERPARAMETERS
# ==========================================
# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 16
learning_rate = 2e-5
num_epochs = 3

In [ ]:

# ==========================================
# 2. LOAD AND PREPARE THE DATASET
# ==========================================
print("Loading dataset...")
# This automatically downloads the deepset/prompt-injections dataset
dataset = load_dataset("deepset/prompt-injections")

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_function(examples):
    # Truncate long prompts to 512 tokens (BERT's maximum)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

In [ ]:
type(dataset)

In [ ]:


print("Tokenizing data...")
# Apply tokenization to the whole dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Remove the raw text column (PyTorch only wants numbers)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
# Rename 'label' to 'labels' (which is what the BERT model expects)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
# Set the format to PyTorch tensors
tokenized_datasets.set_format("torch")

In [ ]:

# Create DataLoaders to feed data to the model in batches
train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=batch_size)
eval_dataloader = DataLoader(tokenized_datasets["test"], batch_size=batch_size)

# ==========================================
# 3. INITIALIZE THE MODEL
# ==========================================
print("Loading pre-trained BERT model...")
# Load BERT with a 2-class classification head (0: Safe, 1: Injection)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

# ==========================================
# 4. OPTIMIZER & SCHEDULER
# ==========================================
# AdamW is the standard optimizer for transformer models
optimizer = AdamW(model.parameters(), lr=learning_rate)

num_training_steps = num_epochs * len(train_dataloader)
# A learning rate scheduler helps the model settle into the optimal weights
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [ ]:
# ==========================================
# 5. THE TRAINING LOOP
# ==========================================
print("Starting training...")
progress_bar = tqdm(range(num_training_steps))

model.train() # Put model in training mode
for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Move all tensors in the batch to the GPU (if available)
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass: Feed the data into the model
        outputs = model(**batch)

        # Calculate loss (how wrong the model was)
        loss = outputs.loss

        # Backward pass: Calculate gradients
        loss.backward()

        # Update weights
        optimizer.step()
        lr_scheduler.step()

        # Clear gradients for the next step
        optimizer.zero_grad()
        progress_bar.update(1)

    print(f"Epoch {epoch + 1} complete.")

# ==========================================
# 6. SAVE YOUR SECURITY GUARDRAIL
# ==========================================
print("Saving fine-tuned model...")
# Save the model and tokenizer to a local directory
model.save_pretrained("./mcp-bert-guardrail")
tokenizer.save_pretrained("./mcp-bert-guardrail")
print("Done! Your model is ready to protect your MCP server.")